# Arranque en Colab

Deja listo el entorno de referencia: una GPU T4, Ollama 0.40.10, el modelo de `config.toml` y los datos del Mundial. Corré las celdas **en orden**. Tarda unos 10 minutos la primera vez, casi todo en bajar el modelo.

**Antes de empezar:** elegí la GPU T4 en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Cambiarla después reemplaza la máquina y borra lo que hicieron las celdas siguientes. Desde VS Code: *Select Kernel → Colab → New Colab Server* y elegí la T4.

El disco de Colab se borra al desconectarse. Sacá `trazas/` y `resultados/` antes de cerrar.

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader || echo "sin GPU: cambiá el entorno a T4" 

Tesla T4, 15360 MiB, 0 MiB


## 1 · Tu código
Poné la URL de **tu fork** (o de tu repositorio privado, ver el README). Si el repositorio es privado, la URL tiene que llevar un token de lectura: `https://<token>@github.com/<usuario>/<repo>.git`.

In [2]:
import os

REPO_URL = "https://github.com/TU_USUARIO/TU_FORK.git"   # <- cambiá esto
CARPETA = "proyecto"

assert "TU_USUARIO" not in REPO_URL, "Poné la URL de tu fork en REPO_URL"
if not os.path.exists(CARPETA):
    !git clone $REPO_URL $CARPETA
os.chdir(CARPETA)
!git pull
!git log --oneline -1

AssertionError: Poné la URL de tu fork en REPO_URL

## 2 · Ollama, con la versión fijada
El instalador baja la versión que le pidas con `OLLAMA_VERSION`. Sin eso baja la última, y los tiempos y el formato de las llamadas a herramientas pueden cambiar.

In [ ]:
OLLAMA_VERSION = "0.40.10"
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=$OLLAMA_VERSION sh

In [ ]:
import os, subprocess, time, requests

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"   # un solo modelo en la placa
os.environ["OLLAMA_NUM_PARALLEL"] = "1"        # un pedido a la vez: tiempos comparables

def ollama_vivo():
    try:
        return requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok
    except requests.exceptions.RequestException:
        return False

if not ollama_vivo():
    # start_new_session: interrumpir una celda no le manda SIGINT al servidor.
    subprocess.Popen(["ollama", "serve"], start_new_session=True,
                     stdout=open("ollama.log", "a"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_vivo():
            break
        time.sleep(1)
print("Ollama está listo." if ollama_vivo() else "El servidor no levantó: mirá ollama.log")

## 3 · El modelo y los datos

In [ ]:
import config

MODELO = config.cargar()["modelo"]["nombre"]
!ollama pull $MODELO
!python datos/descargar.py

## 4 · Chequeos
Tres cosas que conviene mirar antes de medir nada: la versión de Ollama, que el modelo corra **100% en la GPU**, y que la configuración sea válida.

In [ ]:
import llm

cfg = config.cargar()
print("Ollama", llm.version(), "(esperada: " + OLLAMA_VERSION + ")")
if llm.version() != OLLAMA_VERSION:
    print("   ATENCIÓN: no es la versión del kit; decilo en tu informe.")

# Una llamada de prueba para que el modelo quede cargado, y después dónde corre.
respuesta = llm.chat([{"role": "user", "content": "Respondé solo con la palabra: listo"}],
                     modelo=MODELO, num_ctx=cfg["modelo"]["num_ctx"], num_predict=20)
print("respuesta:", respuesta.mensaje["content"].strip(), "·", respuesta.tokens_entrada, "tokens leídos,",
      respuesta.tokens_salida, "escritos,", respuesta.duracion_ms, "ms")
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
if "100% GPU" not in ps:
    print("ATENCIÓN: el modelo no corre 100% en la GPU. Eso cambia los tiempos: revisá ollama.log.")

In [ ]:
!pip install -q pytest
!python -m pytest -q

## 5 · Corré el kit

Con el esqueleto sin implementar, `agente.py` termina con estado `error` y `evaluar.py` da 0 aciertos: es el punto de partida. La línea de base a libro cerrado sí funciona desde ahora.

```
!python libro_cerrado.py --corrida trazas/libro_cerrado
!python evaluar.py --corrida trazas/libro_cerrado

!python correr_todas.py --corrida trazas/mi_corrida
!python evaluar.py --corrida trazas/mi_corrida
```